# Globally Injective Parametrization
Demonstrates flexibility of MeshFEM's IPC integration by implementing a globally injective parametrization using a locally injective parametrization energy and an IPC self-collision barrier energy.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'

In [ ]:
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer
import parametrization, benchmark
import energy
import numpy as np

m = mesh.Mesh('../models/hand.msh')
# m = mesh.Mesh('../models/cow2Disc.msh')

In [ ]:
# Scale so that the surface area is pi (to match [Su et al. 2020])
m.setVertices(m.vertices() * np.sqrt(np.pi / m.volume))

In [ ]:
INCLUDE_CONTACT = False
RECORD_VIDEO = False

In [ ]:
uv = mesh_energy.NodalVars(m, 2)

In [ ]:
# This should really go in a param_utils Python module within MeshFEM...
import igl, numpy as np

def map_vertices_to_circle_area_normalized(V, F, bnd):
    """
    Python equivalent of the C++ function:

        void map_vertices_to_circle_area_normalized(
            const Eigen::MatrixXd& V,
            const Eigen::MatrixXi& F,
            const Eigen::VectorXi& bnd,
            Eigen::MatrixXd& UV)

    Parameters
    ----------
    V : (n, 3) float ndarray
        Vertex positions
    F : (m, 3) int ndarray
        Triangle indices
    bnd : (k,) int ndarray
        Boundary vertex indices

    Returns
    -------
    bc : (k, 2) float ndarray
        UV coordinates for the boundary vertices, placed on a circle
        whose radius is sqrt(mesh_area / pi).
    """
    # 1) Compute total mesh area via doublearea
    #    igl.doublearea(...) returns one "double area" value per face
    dblArea_orig = igl.doublearea(V, F)  # shape (m,)
    area = dblArea_orig.sum() / 2.0
    radius = np.sqrt(area / np.pi)

    # Uncomment if you want the same console output as in C++:
    # print(f"map_vertices_to_circle_area_normalized, area = {area}, radius = {radius}")
    map_ij = np.zeros((V.shape[0], ), dtype=int)
    interior = []
    isOnBnd = np.zeros((V.shape[0], ), dtype=bool)
    for i in range(bnd.shape[0]):
        isOnBnd[bnd[i]] = True
        map_ij[bnd[i]] = i
    for i in range(isOnBnd.shape[0]):
        if (not isOnBnd[i]):
            map_ij[i] = len(interior)
            interior.append(i)

    # 2) Build a running length array along boundary vertices
    k = bnd.shape[0]
    length = np.zeros(k)
    for i in range(1, k):
        prev_idx = bnd[i - 1]
        curr_idx = bnd[i]
        length[i] = length[i - 1] + np.linalg.norm(V[prev_idx] - V[curr_idx])

    # Add the distance between the last and the first boundary vertex
    total_len = length[-1] + np.linalg.norm(V[bnd[0]] - V[bnd[-1]])

    # 3) Place boundary vertices along the circle of computed radius
    bc = np.zeros((k, 2))
    for i in range(k):
        frac = length[i] * (2.0 * np.pi) / total_len
        bc[map_ij[bnd[i]], 0] = radius * np.cos(frac)
        bc[map_ij[bnd[i]], 1] = radius * np.sin(frac)
        # bc[i, 0] = radius * np.cos(frac)
        # bc[i, 1] = radius * np.sin(frac)
    return bc

def getBDdataOnNormalizedCircle(m):
    BV = m.boundaryVertices()
    bnd_loop = igl.boundary_loop(m.elements())
    bloop = np.searchsorted(BV, bnd_loop)
    bdry_uv = map_vertices_to_circle_area_normalized(m.vertices(), m.elements(), bnd_loop)
    bdry_uv[bloop] =  bdry_uv.copy()
    return bdry_uv

def tutteInitialization(m, bdry_uv):
    # Tutte Initialization
    uv_init = parametrization.harmonic(m, bdry_uv, False)
    flip_list = parametrization.getFlips(m, uv_init)
    if len(flip_list) > 0:  uv_init = parametrization.harmonic(m, bdry_uv, True)
    return uv_init

In [ ]:
e = energy.SymmetricDirichletDerivativeFree(2)

In [ ]:
e = energy.SymmetricDirichlet(2)

In [ ]:
uv_init = tutteInitialization(m, getBDdataOnNormalizedCircle(m))

In [ ]:
# Initialization: tutte embedding
# uv.setVars(parametrization.lscm(m).ravel())
uv.setVars(uv_init.ravel())

In [ ]:
param = mesh_energy.Parametrization(m, uv, e)
objectives = [param]

if INCLUDE_CONTACT:
    import meshfem_ipc
    cm = meshfem_ipc.CollisionMesh(m, embeddingDimension=2)
    contact = meshfem_ipc.IPCObjectiveTerm(uv, cm)
    contact.useAdaptiveBarrier = True
    objectives.append(contact)
    
    contact.ccdTol = 1e-3

In [ ]:
# contact.barrierStiffness = 1e5

In [ ]:
# contact.dhat = 0.0001

In [ ]:
# Construct parametrization energy and problem
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, objectives)

In [ ]:
import flip_avoiding_step_length

In [ ]:
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())

In [ ]:
em = MeshFEM.EmbeddedMesh(m, uv)
v = viewer.Viewer(em, wireframe=True)
v.show()

In [ ]:
param.objective()

In [ ]:
# Work around energy nullspace by adding a small shift
prob.hessianShift = 1e-10
prob.useRelativeHessianShift = True
opt = prob.optimizer()
opt.options.niter = 500

In [ ]:
opt.options.gradTol = 0.1

In [ ]:
prob.setCustomIterationCallback(v.updater(10))

In [ ]:
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 2

In [ ]:
# opt.options.factorizer = opt.options.factorizer.CatamariAMD

In [ ]:
if RECORD_VIDEO:
    prob.setCustomIterationCallback(v.updater())
    name = 'globally_injective' if INCLUDE_CONTACT else 'not_globally_injective'
    v.recordStart(f'{name}.mp4', renderScale=4, outputScale=2, lineWidthScale=0.5)

In [ ]:
opt.options.gradTol = 1e-6

In [ ]:
benchmark.reset()
rep = opt.optimize()
benchmark.report()